# Diseño de Laboratorio — Rutas Mínimas (Dijkstra & Bellman-Ford)

**Grupo:** DATAFORGE  

**Integrantes:**
- Enrique Jara Escobar
- Sergio Alejandro Ariza
- Andres Ricardo Poveda

**Semana:** 16 · Diseño de Datos y Algoritmos  

---

## 1) Descripción del Problema

Se tienen tres grafos dirigidos y ponderados donde cada nodo representa un punto de
la red y cada arista tiene un peso (costo de transitar ese tramo).  
El objetivo es determinar, para cada grafo, cuál es el peso total de la ruta más corta
que va del nodo inicio al nodo fin.

| Grafo | Nodos | Aristas | Pesos negativos | Algoritmo |
|-------|-------|---------|-----------------|--------------------|
| 1     | 6 (0-5) | 9     | No              | Dijkstra |
| 2     | 5 (0-4) | 5     | No              | Dijkstra |
| 3     | 5 (0-4) | 6     | Sí (−1)         | Bellman-Ford |

### Modelo general

- $G = (V, E)$ — grafo dirigido con conjunto de vértices $V$ y aristas $E$.
- $w : E \to \mathbb{R}$ — función de peso por arista.
- Inicio $s$ — nodo fuente; Fin $t$ — nodo destino.
- Objetivo: hallar $d(s, t) = \min_{p \in P(s,t)} \sum_{e \in p} w(e)$
  donde $P(s,t)$ es el conjunto de todos los caminos de $s$ a $t$.

### Técnica de relajación

Ambos algoritmos (Dijkstra y Bellman-Ford) se basan en relajar aristas:  
dado un nodo $u$ con distancia estimada $d[u]$ y una arista $(u, v, w)$,

$$\text{si } d[u] + w < d[v] \implies d[v] \leftarrow d[u] + w$$

La diferencia está en cuándo y cuántas veces se aplica esa relajación.


## 2) Requerimientos

### Requerimientos Funcionales

- **RF01** — Representación del grafo: El sistema debe representar cada grafo como un diccionario de listas de adyacencia {nodo: [(vecino, peso), ...]} sin usar clases.
- **RF02** — Relajación Dijkstra: Para grafos sin pesos negativos, el algoritmo debe aplicar relajación de aristas usando una cola de mínimos (heapq) garantizando que cada nodo se procese una sola vez.
- **RF03** — Relajación Bellman-Ford: Para grafos con pesos potencialmente negativos, el algoritmo debe relajar todas las aristas $|V|-1$ veces y detectar ciclos negativos.
- **RF04** — Reconstrucción del camino: El sistema debe registrar los nodos padre durante la relajación y reconstruir la ruta óptima de inicio a fin.
- **RF05** — Presentación de resultados: El sistema debe mostrar el costo mínimo y la secuencia de nodos del camino óptimo para cada grafo.

### Requerimientos No Funcionales

- **RNF01** — Eficiencia Dijkstra: El algoritmo debe ejecutarse en $O((V + E) \log V)$ usando cola de prioridad con monticulos min-heap.
- **RNF02** — Eficiencia Bellman-Ford: El algoritmo debe ejecutarse en $O(V \cdot E)$, iterando $|V|-1$ rondas de relajación.
- **RNF03** — Sin librerías externas de grafos: El código usa únicamente Python puro y heapq de la biblioteca estándar.
- **RNF04** — Sin clases: La implementación utiliza funciones y estructuras de datos nativas (dict, list).
- **RNF05** — Portabilidad: Compatible con Python 3 sin instalaciones adicionales.
- **RNF06** — Detección de ciclos negativos: Bellman-Ford debe alertar si existe un ciclo de peso negativo en el grafo.


## 3) Historias de Usuario

- **HU01** — Como estudiante, quiero representar cada grafo del enunciado como un diccionario de Python para poder validar visualmente las conexiones antes de correr el algoritmo. Criterio: el diccionario contiene exactamente los nodos y aristas del diagrama con sus pesos correctos. (RF01)
- **HU02** — Como analista, quiero ejecutar Dijkstra sobre grafos sin pesos negativos para obtener la ruta mínima en el menor tiempo posible. Criterio: el algoritmo termina en $O((V+E)\log V)$ y devuelve el costo y camino correctos. (RF02)
- **HU03** — Como analista, quiero ejecutar Bellman-Ford sobre el Grafo 3 (que tiene una arista de peso −1) para obtener el camino mínimo sin restricciones de signo. Criterio: el algoritmo itera $|V|-1$ veces, relaja correctamente el peso −1 y devuelve la distancia mínima. (RF03)
- **HU04** — Como usuario, quiero ver impresa la secuencia de nodos del camino óptimo para cada grafo para poder verificar la ruta a mano. Criterio: la reconstrucción sigue los punteros padre desde fin hasta inicio e invierte la lista. (RF04)
- **HU05** — Como supervisor, quiero ver en pantalla el peso total de cada ruta mínima para responder directamente la pregunta del enunciado. Criterio: el sistema imprime Costo mínimo: X con el valor numérico correcto. (RF05)


## 4) Análisis de Complejidad

### Dijkstra (Grafos 1 y 2)

| Operación | Mejor Caso | Caso Promedio | Peor Caso | Razón |
|-----------|-----------|---------------|-----------|-------|
| Inicializar distancias | $O(V)$ | $O(V)$ | $O(V)$ | Recorre todos los vértices |
| Extraer mínimo (heapq) | $O(\log V)$ | $O(\log V)$ | $O(\log V)$ | Operación sobre heap |
| Relajar arista | $O(1)$ | $O(1)$ | $O(\log V)$ | Puede requerir inserción en heap |
| Relajar todas las aristas | $O(E \log V)$ | $O(E \log V)$ | $O(E \log V)$ | $E$ relajaciones × costo heap |
| **Total Dijkstra** | **$O((V+E)\log V)$** | **$O((V+E)\log V)$** | **$O((V+E)\log V)$** | |

### Bellman-Ford (Grafo 3)

| Operación | Mejor Caso | Caso Promedio | Peor Caso | Razón |
|-----------|-----------|---------------|-----------|-------|
| Inicializar distancias | $O(V)$ | $O(V)$ | $O(V)$ | Un valor por vértice |
| Ronda de relajación (todas las aristas) | $O(E)$ | $O(E)$ | $O(E)$ | Recorre $E$ aristas |
| Número de rondas | $O(1)$\* | $O(V)$ | $O(V)$ | \*Termina antes si no hay cambios |
| Verificación ciclo negativo | $O(E)$ | $O(E)$ | $O(E)$ | Una última pasada |
| **Total Bellman-Ford** | **$O(V \cdot E)$** | **$O(V \cdot E)$** | **$O(V \cdot E)$** | |

### Complejidad Espacial

| Estructura | Espacio |
|------------|---------|
| Diccionario de adyacencia | $O(V + E)$ |
| Arreglo de distancias `dist` | $O(V)$ |
| Diccionario de padres | $O(V)$ |
| Cola de prioridad (Dijkstra) | $O(E)$ en el peor caso |
| **Total espacial** | **$O(V + E)$** |

## 5) Diagrama de Flujo

### Dijkstra

![alt text](</workspaces/Diseno-de-Datos-y-Algoritmos/Semana 16/Anexos/Diagrama Flujo.png>)

### Bellman-Ford

![alt text](</workspaces/Diseno-de-Datos-y-Algoritmos/Semana 16/Anexos/Diagrama Bellman.png>)


## 6) Funciones Base de Relajación


In [1]:
import heapq

def dijkstra(grafo, inicio):
    dist  = {n: float('inf') for n in grafo}
    padre = {n: None         for n in grafo}
    dist[inicio] = 0
    cola = [(0, inicio)]
    while cola:
        d, u = heapq.heappop(cola)
        if d > dist[u]:
            continue
        for v, peso in grafo[u]:
            nueva = dist[u] + peso
            if nueva < dist[v]:             # RELAJACIÓN
                dist[v]  = nueva
                padre[v] = u
                heapq.heappush(cola, (nueva, v))
    return dist, padre


def bellman_ford(aristas, nodos, inicio):
    dist  = {n: float('inf') for n in nodos}
    padre = {n: None         for n in nodos}
    dist[inicio] = 0
    for _ in range(len(nodos) - 1):
        for u, v, peso in aristas:
            if dist[u] + peso < dist[v]:    # RELAJACIÓN
                dist[v]  = dist[u] + peso
                padre[v] = u
    for u, v, peso in aristas:
        if dist[u] + peso < dist[v]:
            raise ValueError("Ciclo de peso negativo detectado")
    return dist, padre


def reconstruir_camino(padre, inicio, fin):
    camino, nodo = [], fin
    while nodo is not None:
        camino.append(nodo)
        nodo = padre[nodo]
    camino.reverse()
    return camino if camino and camino[0] == inicio else []


print("Funciones de relajacion cargadas.")


Funciones de relajacion cargadas.


## 7) Ejercicio 1 — Grafo de 6 Nodos (Dijkstra)

```
       ──4──▶ [3]
      ↑          ╲3
 5  [1]    6      ▼
[0]──▶      ──────▶ [5] fin
 ╲   ╲ 2↗        ↗
  2    ▶[4]──1───
   ↘  ↗
   [2]──7──▶[4]
```

| Arista | Peso |
|--------|------|
| 0 → 1  | 5    |
| 0 → 2  | 2    |
| 1 → 3  | 4    |
| 1 → 4  | 2    |
| 2 → 1  | 8    |
| 2 → 4  | 7    |
| 3 → 5  | 3    |
| 3 → 4  | 6    |
| 4 → 5  | 1    |

**inicio = 0 · fin = 5**

In [ ]:
grafo1 = {
    0: [(1, 5), (2, 2)],
    1: [(3, 4), (4, 2)],
    2: [(1, 8), (4, 7)],
    3: [(5, 3), (4, 6)],
    4: [(5, 1)],
    5: [],
}

INICIO1, FIN1 = 0, 5
dist1, padre1 = dijkstra(grafo1, INICIO1)
camino1       = reconstruir_camino(padre1, INICIO1, FIN1)

print("=" * 40)
print("  GRAFO 1 — Dijkstra")
print("=" * 40)
for nodo, d in sorted(dist1.items()):
    marca = " ← FIN" if nodo == FIN1 else ""
    print(f"  Nodo {nodo}: {d}{marca}")
print()
print(f"  Camino : {' → '.join(str(n) for n in camino1)}")
print(f"  Costo  : {dist1[FIN1]}")
print("=" * 40)


### Traza de Relajación — Grafo 1

| Paso | Nodo extraído | dist[0] | dist[1] | dist[2] | dist[3] | dist[4] | dist[5] |
|------|--------------|---------|---------|---------|---------|---------|--------|
| 0    | —            | **0**   | ∞       | ∞       | ∞       | ∞       | ∞      |
| 1    | 0 (d=0)      | 0       | **5**   | **2**   | ∞       | ∞       | ∞      |
| 2    | 2 (d=2)      | 0       | 5       | 2       | ∞       | **9**   | ∞      |
| 3    | 1 (d=5)      | 0       | 5       | 2       | **9**   | **7**   | ∞      |
| 4    | 4 (d=7)      | 0       | 5       | 2       | 9       | 7       | **8**  |
| 5    | 3 (d=9)      | 0       | 5       | 2       | 9       | 7       | 8 (sin mejora) |
| 6    | 5 (d=8)      | 0       | 5       | 2       | 9       | 7       | **8**  |

**Resultado:** ruta `0 → 1 → 4 → 5` con costo **8**.

## 8) Ejercicio 2 — Grafo de 5 Nodos (Dijkstra)

```
        ──20──▶ [2]
       ↑              ╲30
 10  [1]──1──▶[3]─1─▶  ▼
[0]──▶                [4] fin
```

| Arista | Peso |
|--------|------|
| 0 → 1  | 10   |
| 1 → 2  | 20   |
| 1 → 3  | 1    |
| 3 → 2  | 1    |
| 2 → 4  | 30   |

**inicio = 0 · fin = 4**

In [ ]:
grafo2 = {
    0: [(1, 10)],
    1: [(2, 20), (3, 1)],
    2: [(4, 30)],
    3: [(2, 1)],
    4: [],
}

INICIO2, FIN2 = 0, 4
dist2, padre2 = dijkstra(grafo2, INICIO2)
camino2       = reconstruir_camino(padre2, INICIO2, FIN2)

print("=" * 40)
print("  GRAFO 2 — Dijkstra")
print("=" * 40)
for nodo, d in sorted(dist2.items()):
    marca = " ← FIN" if nodo == FIN2 else ""
    print(f"  Nodo {nodo}: {d}{marca}")
print()
print(f"  Camino : {' → '.join(str(n) for n in camino2)}")
print(f"  Costo  : {dist2[FIN2]}")
print("=" * 40)


### Traza de Relajación — Grafo 2

| Paso | Nodo extraído | dist[0] | dist[1] | dist[2] | dist[3] | dist[4] |
|------|--------------|---------|---------|---------|---------|--------|
| 0    | —            | **0**   | ∞       | ∞       | ∞       | ∞      |
| 1    | 0 (d=0)      | 0       | **10**  | ∞       | ∞       | ∞      |
| 2    | 1 (d=10)     | 0       | 10      | **30**  | **11**  | ∞      |
| 3    | 3 (d=11)     | 0       | 10      | **12**  | 11      | ∞      |
| 4    | 2 (d=12)     | 0       | 10      | 12      | 11      | **42** |
| 5    | 4 (d=42)     | 0       | 10      | 12      | 11      | **42** |

**Resultado:** ruta `0 → 1 → 3 → 2 → 4` con costo **42**.

## 9) Ejercicio 3 — Grafo con Peso Negativo (Bellman-Ford)

```
        ──2──▶ [2] ──2──▶ [4] fin
       ↗            ╲2
 2  [0]              ▼
    ╲──2──▶ [1] ←─(−1)─ [3] ──2──▶ [4]
```

| Arista | Peso |
|--------|------|
| 0 → 2  | 2    |
| 0 → 1  | 2    |
| 2 → 4  | 2    |
| 2 → 3  | 2    |
| 3 → 1  | −1   |
| 3 → 4  | 2    |

⚠️ La arista `3 → 1` tiene peso **−1**, por lo que se usa **Bellman-Ford**.  
**inicio = 0 · fin = 4**

In [ ]:
nodos3   = [0, 1, 2, 3, 4]
aristas3 = [
    (0, 2,  2),
    (0, 1,  2),
    (2, 4,  2),
    (2, 3,  2),
    (3, 1, -1),   # peso negativo → Bellman-Ford
    (3, 4,  2),
]

INICIO3, FIN3 = 0, 4
dist3, padre3 = bellman_ford(aristas3, nodos3, INICIO3)
camino3       = reconstruir_camino(padre3, INICIO3, FIN3)

print("=" * 40)
print("  GRAFO 3 — Bellman-Ford")
print("=" * 40)
for nodo in sorted(nodos3):
    marca = " ← FIN" if nodo == FIN3 else ""
    print(f"  Nodo {nodo}: {dist3[nodo]}{marca}")
print()
print(f"  Camino : {' → '.join(str(n) for n in camino3)}")
print(f"  Costo  : {dist3[FIN3]}")
print("=" * 40)


### Traza de Relajación — Grafo 3 (Bellman-Ford)

| Ronda | Arista relajada | dist[0] | dist[1] | dist[2] | dist[3] | dist[4] |
|-------|----------------|---------|---------|---------|---------|--------|
| 0     | —              | **0**   | ∞       | ∞       | ∞       | ∞      |
| 1     | 0→2 (w=2)      | 0       | ∞       | **2**   | ∞       | ∞      |
| 1     | 0→1 (w=2)      | 0       | **2**   | 2       | ∞       | ∞      |
| 1     | 2→4 (w=2)      | 0       | 2       | 2       | ∞       | **4**  |
| 1     | 2→3 (w=2)      | 0       | 2       | 2       | **4**   | 4      |
| 1     | 3→1 (w=−1)     | 0       | **3**   | 2       | 4       | 4      |
| 1     | 3→4 (w=2)      | 0       | 3       | 2       | 4       | 4 (sin mejora) |
| 2     | todas          | 0       | 3       | 2       | 4       | 4 (sin cambios) |
| 3     | todas          | 0       | 3       | 2       | 4       | 4 (sin cambios) |
| 4     | todas          | 0       | 3       | 2       | 4       | 4 (sin cambios) |

**Resultado:** ruta `0 → 2 → 4` con costo **4**.  
*(La arista 3→1 con peso −1 mejora dist[1] de 2 a 3... pero 1 no conecta a 4, por lo que no impacta el camino óptimo al fin.)*

## 11) Tests

In [ ]:
pass_count = 0
fail_count = 0

def check(nombre, obtenido, esperado):
    global pass_count, fail_count
    if obtenido == esperado:
        print(f"   PASS  {nombre}")
        pass_count += 1
    else:
        print(f"   FAIL  {nombre}")
        print(f"         Esperado : {esperado}")
        print(f"         Obtenido : {obtenido}")
        fail_count += 1

print("=" * 55)
print("  TESTS — Dijkstra & Bellman-Ford")
print("=" * 55)

check("T01 — Grafo 1: costo = 8",           dist1[5],   8)
check("T02 — Grafo 1: camino = [0,1,4,5]",  camino1,    [0, 1, 4, 5])
check("T03 — Grafo 1: dist[2] = 2",         dist1[2],   2)
check("T04 — Grafo 1: dist[4] = 7",         dist1[4],   7)
check("T05 — Grafo 2: costo = 42",          dist2[4],   42)
check("T06 — Grafo 2: camino = [0,1,3,2,4]",camino2,   [0, 1, 3, 2, 4])
check("T07 — Grafo 2: dist[3] = 11",        dist2[3],   11)
check("T08 — Grafo 2: dist[2] = 12",        dist2[2],   12)
check("T09 — Grafo 3: costo = 4",           dist3[4],   4)
check("T10 — Grafo 3: camino = [0,2,4]",    camino3,    [0, 2, 4])
check("T11 — Grafo 3: dist[2] = 2",         dist3[2],   2)
check("T12 — Grafo 3: dist[3] = 4",         dist3[3],   4)
check("T13 — Grafo 3: peso −1 → dist[1]=3", dist3[1],   3)

_d, _ = dijkstra({0: []}, 0)
check("T14 — Dijkstra: nodo único dist[0]=0", _d[0], 0)

_d2, _ = bellman_ford([], [0, 1], 0)
check("T15 — Bellman-Ford: sin aristas dist[1]=inf", _d2[1], float('inf'))

check("T16 — reconstruir sin ruta = []",
      reconstruir_camino({0: None, 1: None}, 0, 1), [])

_d3, _ = dijkstra(grafo1, 5)
check("T17 — Dijkstra: inicio=fin → dist=0", _d3[5], 0)

try:
    bellman_ford([(0,1,-1),(1,0,-1)], [0,1], 0)
    check("T18 — BF detecta ciclo negativo", False, True)
except ValueError:
    check("T18 — BF detecta ciclo negativo", True, True)

print("=" * 55)
print(f"  Resultado: {pass_count} PASS  |  {fail_count} FAIL")
print("=" * 55)
